# Set up

## Load libraries

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns; sns.set()
from matplotlib import pyplot as plt

# Sample
## Get URIs
import awswrangler as wr
## Run
import pathlib
import sagemaker
from sagemaker import sklearn as srsn

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


## Declare constants

In [2]:
# Sample
## Get files
data_uri_prefix_sr = 's3://20231010-gen-xii/01_ad/01_data_prep/05_leaky_features/04_write_dfs'
## Run
frac_ft = 1e-2

# Sample

## Get URIs

In [3]:
data_uris_ss = pd.Series(data=wr.s3.list_objects(path=data_uri_prefix_sr))

print(*data_uris_ss, sep='\n')

s3://20231010-gen-xii/01_ad/01_data_prep/05_leaky_features/04_write_dfs/df_test_noleaks.gzip
s3://20231010-gen-xii/01_ad/01_data_prep/05_leaky_features/04_write_dfs/df_train_noleaks.gzip
s3://20231010-gen-xii/01_ad/01_data_prep/05_leaky_features/04_write_dfs/df_valid_noleaks.gzip


## Write script

In [4]:
%%file script.py

import numpy as np
import pandas as pd
# import seaborn as sns; sns.set()
# from matplotlib import pyplot as plt

import pathlib

import argparse
import logging

if __name__ == '__main__':
    # Configure logger
    logging.basicConfig(format='%(levelname)s - %(asctime)s - %(message)s', level=logging.INFO)
    logging.info(msg='Configure logger')
    
    # Parse arguments
    ap = argparse.ArgumentParser()
    # Read in data
    ap.add_argument('--file_ph', type=str)
    # Sample
    ap.add_argument('--frac_ft', type=float)
    args_ns, _ = ap.parse_known_args()
    logging.info(msg='Parse arguments')
    
    # Assign
    # Read in data
    file_ph = pathlib.Path(args_ns.file_ph)
    # Sample
    frac_ft = args_ns.frac_ft
    logging.info(msg='Assign')
    
    # Read in data
    base_directory_sr = '/opt/ml/processing'
    input_directory_sr = f'{base_directory_sr}/input'
    split_sr = file_ph.stem.split('_')[1]
    df = pd.read_parquet(path=f'{input_directory_sr}/{split_sr}/{file_ph}')
    logging.info(msg='Read in data')
    
    # Sample
    print(f'Shape: {df.shape}')
    df = df.sample(frac=frac_ft, random_state=0)
    print(f'Shape: {df.shape}')
    logging.info(msg='Sample')
    
    # Write
    output_directory_sr = f'{base_directory_sr}/output'
    file_sr = f'{file_ph.stem}_raw_{str(int(frac_ft * 1e2))}{file_ph.suffix}'
    df.to_parquet(path=f'{output_directory_sr}/{split_sr}/{file_sr}', compression='gzip')
    logging.info(msg='Write')

Overwriting script.py


## Run

In [5]:
sklp = srsn.SKLearnProcessor(
    framework_version='1.2-1', 
    role=sagemaker.get_execution_role(), 
    instance_count=1, 
    instance_type='ml.m5.4xlarge')
base_directory_sr = '/opt/ml/processing'
input_directory_sr = f'{base_directory_sr}/input'
output_directory_sr = f'{base_directory_sr}/output'

for index_it, data_uri_sr in enumerate(iterable=data_uris_ss):
    file_sr = pathlib.Path(data_uri_sr).name
    split_sr = file_sr.split('_')[1]
    sklp.run(
        code='script.py', 
        inputs=[sagemaker.processing.ProcessingInput(
            source=data_uri_sr, destination=f'{input_directory_sr}/{split_sr}')],
        outputs=[sagemaker.processing.ProcessingOutput(
            source=f'{output_directory_sr}/{split_sr}', destination=data_uri_prefix_sr)], 
        arguments=[
            # Read in data
            '--file_ph', file_sr,
            # Sample
            '--frac_ft', str(frac_ft)])

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


INFO:sagemaker:Creating processing-job with name sagemaker-scikit-learn-2023-10-12-17-34-02-976


..............................INFO - 2023-10-12 17:39:05,939 - Configure logger
INFO - 2023-10-12 17:39:05,940 - Parse arguments
INFO - 2023-10-12 17:39:05,940 - Assign
INFO - 2023-10-12 17:39:07,880 - Read in data
Shape: (420140, 1566)
Shape: (4201, 1566)
INFO - 2023-10-12 17:39:08,006 - Sample
INFO - 2023-10-12 17:39:08,902 - Write



INFO:sagemaker:Creating processing-job with name sagemaker-scikit-learn-2023-10-12-17-39-59-024


....................................INFO - 2023-10-12 17:45:58,509 - Configure logger
INFO - 2023-10-12 17:45:58,510 - Parse arguments
INFO - 2023-10-12 17:45:58,510 - Assign
INFO - 2023-10-12 17:46:06,922 - Read in data
Shape: (1260418, 1566)
Shape: (12604, 1566)
INFO - 2023-10-12 17:46:07,320 - Sample
INFO - 2023-10-12 17:46:09,640 - Write



INFO:sagemaker:Creating processing-job with name sagemaker-scikit-learn-2023-10-12-17-46-57-231


...............................INFO - 2023-10-12 17:52:06,394 - Configure logger
INFO - 2023-10-12 17:52:06,395 - Parse arguments
INFO - 2023-10-12 17:52:06,395 - Assign
INFO - 2023-10-12 17:52:08,244 - Read in data
Shape: (420140, 1566)
Shape: (4201, 1566)
INFO - 2023-10-12 17:52:08,372 - Sample
INFO - 2023-10-12 17:52:09,230 - Write



## Get URIs

In [6]:
data_uris_ss = pd.Series(data=wr.s3.list_objects(path=data_uri_prefix_sr))

print(*data_uris_ss, sep='\n')

s3://20231010-gen-xii/01_ad/01_data_prep/05_leaky_features/04_write_dfs/df_test_noleaks.gzip
s3://20231010-gen-xii/01_ad/01_data_prep/05_leaky_features/04_write_dfs/df_test_noleaks_raw_1.gzip
s3://20231010-gen-xii/01_ad/01_data_prep/05_leaky_features/04_write_dfs/df_train_noleaks.gzip
s3://20231010-gen-xii/01_ad/01_data_prep/05_leaky_features/04_write_dfs/df_train_noleaks_raw_1.gzip
s3://20231010-gen-xii/01_ad/01_data_prep/05_leaky_features/04_write_dfs/df_valid_noleaks.gzip
s3://20231010-gen-xii/01_ad/01_data_prep/05_leaky_features/04_write_dfs/df_valid_noleaks_raw_1.gzip
